# Notebook 06A — Model C Extended-Training Probe

**Status:** exploratory follow-on to the frozen Notebook 05 A/B/C experiment.

This notebook asks one narrow question: **was Model C still training-duration constrained at epoch 3 / update 3,663, or does continued training reach validation saturation or begin to overfit?**

The original A/B/C comparison is immutable. The frozen Model C point remains validation loss **3.684501**, perplexity **39.83**, at update **3,663**. Nothing produced here may replace that point in Notebook 05.

## 1. D-095 — Experimental contract

The extension resumes the **exact resumable Model C production checkpoint** at epoch 3 / optimizer update 3,663. It preserves the tokenizer, exact 20M-token corpus, architecture, context/packing, effective batch semantics, optimizer parameter groups, dropout, gradient clipping, validation split/procedure, and numerical semantics.

Only two things change relative to the frozen experiment: (1) training is allowed to continue beyond three epochs, and (2) because the original cosine schedule has ended, the extension uses the terminal learning rate **2e-4 as a constant LR** rather than restarting or retuning the schedule.

Validation remains every 200 optimizer updates plus epoch end. Training loss and validation loss/perplexity must be retained over time in a separate 06A artifact namespace.

**Early stopping:** minimize validation loss, `min_delta=0.001`, `patience=6` consecutive non-improving validation observations. A single increase never stops training.

**Safety ceiling:** at most 10 additional epochs. If validation is still improving at the ceiling, the conclusion is *no saturation observed within the tested range*.

The detailed canonical decision is recorded in `docs/decisions/06a_model_c_extended_training_probe.md`.

In [ ]:
from dataclasses import dataclass

@dataclass(frozen=True)
class ExtendedTrainingContract:
    start_epoch: int = 3
    start_update: int = 3663
    frozen_val_loss: float = 3.684501
    frozen_val_ppl: float = 39.83
    model_parameters: int = 33_497_600
    context_length: int = 512
    examples_per_epoch: int = 39_062
    scored_targets_per_epoch: int = 19_999_744
    updates_per_epoch: int = 1_221
    effective_batch_targets: int = 16_384
    extension_lr: float = 2e-4
    validation_every_updates: int = 200
    validation_targets: int = 256_512
    min_delta: float = 0.001
    patience: int = 6
    max_additional_epochs: int = 10

CONTRACT = ExtendedTrainingContract()
CONTRACT

In [ ]:
# D-095 internal consistency gate — no training or checkpoint mutation occurs here.
max_additional_updates = CONTRACT.max_additional_epochs * CONTRACT.updates_per_epoch
max_global_update = CONTRACT.start_update + max_additional_updates

assert CONTRACT.start_update == 3 * CONTRACT.updates_per_epoch
assert max_additional_updates == 12_210
assert max_global_update == 15_873
assert CONTRACT.extension_lr == 2e-4
assert CONTRACT.patience > 1, 'Early stopping must not stop on the first increase.'
assert CONTRACT.min_delta > 0
assert CONTRACT.validation_targets == 256_512
assert CONTRACT.model_parameters == 33_497_600

print('D-095 contract arithmetic: PASS')
print(f'Frozen start: epoch {CONTRACT.start_epoch}, update {CONTRACT.start_update:,}')
print(f'Extension ceiling: +{CONTRACT.max_additional_epochs} epochs / +{max_additional_updates:,} updates')
print(f'Max global update: {max_global_update:,}')
print(f'Early stopping: min_delta={CONTRACT.min_delta}, patience={CONTRACT.patience}')
print('Frozen A/B/C comparison mutation: NOT PERMITTED')

## 2. D-096 — Hard Resume & Provenance Gate

D-096 is intentionally a **fail-closed gate**, not a training cell. No extension optimizer update may occur until every check below passes.

The critical behavioral identity check is first: after loading the exact update-3,663 checkpoint, rerun the full D-072 validation pass and require the resumed model to reproduce the frozen Model C validation loss **3.684501** to at least five decimal places under the canonical environment. Metadata agreement alone is insufficient.

The gate also proves AdamW moment continuity, scaler/RNG/config identity, canonical corpus/tokenizer hashes, epoch-4 shuffle continuity, and the presence on GitHub `main` of Notebook 05's canonical evaluation artifacts.

D-096 additionally precommits the downstream policy: once the best 06A validation-selected checkpoint is frozen, it receives **exactly one exploratory test evaluation** using the D-092 procedure and the **same D-091 validation prompts/decoding** for qualitative before/after comparison. Neither result may alter training or the frozen Notebook 05 tables.

In [ ]:
# D-096 constants and repository prerequisites. This cell performs no training.
from pathlib import Path
import math

EXPECTED_TOKENIZER_SHA256 = '6ec601a267cec7c843df47927f53c4dd108c85a1d059318aeec4442c7274604f'
EXPECTED_CORPUS_SHA256 = '4101d5b18c38558a58110f54a161763186ab5318111366486ebbfa0a3fe584fa'
EXPECTED_MANIFEST_SHA256 = '4a00196b39311a6c2e2790780e8fc43316f24a014d3d3649028b10a671f8d3fe'
EXPECTED_RESUME_UPDATE = 3_663
EXPECTED_RESUME_EPOCH = 3
EXPECTED_RESUME_VAL_LOSS = 3.684501
RESUME_VAL_DECIMAL_PLACES = 5
EPOCH4_SHUFFLE_SEED = 42 + 4

NB05_REQUIRED = [
    Path('results/evaluation/evidence/validation_history_canonical.csv'),
    Path('results/evaluation/evidence/validation_history_ingestion_audit.json'),
    Path('results/evaluation/evidence/final_test_stream_audit.json'),
    Path('results/evaluation/analysis/final_evaluation_summary.json'),
]

missing_nb05 = [str(p) for p in NB05_REQUIRED if not p.exists()]
assert not missing_nb05, (
    'D-096 BLOCKED: canonical Notebook 05 artifacts are not committed/present: ' + ', '.join(missing_nb05)
)
print('D-096 Notebook 05 artifact prerequisite: PASS')

### 2.1 Behavioral checkpoint identity — reproduce 3.684501 before update 3,664

After the checkpoint/model/validation stream are loaded using the canonical Notebook 04 machinery, run the full deterministic D-072 validation pass **before changing LR or taking any optimizer step**. Record the observed full-precision loss, derived perplexity, absolute difference, and pass/fail state.

The assertion below deliberately uses decimal agreement rather than a loose percentage tolerance. If the canonical T4/FP16 environment produces a small nondeterministic discrepancy that fails this threshold, stop and investigate; do not relax the threshold after seeing the value.

In [ ]:
# EXPECTS: resumed_model and canonical validation loader/evaluator from the frozen D-072 implementation.
# Replace `evaluate_validation_loss(...)` only with the exact reusable D-072 evaluation function.
# observed_resume_val_loss = evaluate_validation_loss(resumed_model, validation_loader)

def assert_resume_validation_identity(observed_resume_val_loss: float) -> dict:
    observed_resume_val_loss = float(observed_resume_val_loss)
    absolute_difference = abs(observed_resume_val_loss - EXPECTED_RESUME_VAL_LOSS)
    rounded_match = (
        round(observed_resume_val_loss, RESUME_VAL_DECIMAL_PLACES)
        == round(EXPECTED_RESUME_VAL_LOSS, RESUME_VAL_DECIMAL_PLACES)
    )
    result = {
        'expected_validation_loss': EXPECTED_RESUME_VAL_LOSS,
        'observed_validation_loss': observed_resume_val_loss,
        'absolute_difference': absolute_difference,
        'required_decimal_places': RESUME_VAL_DECIMAL_PLACES,
        'observed_perplexity': math.exp(observed_resume_val_loss),
        'passed': rounded_match,
    }
    assert rounded_match, (
        f'D-096 BLOCKED: resumed checkpoint validation loss {observed_resume_val_loss:.9f} '
        f'does not reproduce frozen 3.684501 to {RESUME_VAL_DECIMAL_PLACES} decimals.'
    )
    return result

# REQUIRED EXECUTION BEFORE TRAINING:
# resume_validation_audit = assert_resume_validation_identity(observed_resume_val_loss)


### 2.2 AdamW moment continuity

A checkpoint that reloads weights but silently resets AdamW is not a continuation. Every parameter carrying optimizer state must have step **3,663** and non-zero `exp_avg` / `exp_avg_sq` tensors. The audit records counts and fails on any missing, zero, malformed, or wrong-step state.

In [ ]:
def _optimizer_step_as_int(step_value):
    if hasattr(step_value, 'item'):
        return int(step_value.item())
    return int(step_value)

def audit_adamw_state(optimizer, model) -> dict:
    trainable = [p for p in model.parameters() if p.requires_grad]
    audit = {
        'trainable_parameter_tensors': len(trainable),
        'states_checked': 0,
        'missing_or_malformed_states': 0,
        'step_mismatches': 0,
        'zero_exp_avg_tensors': 0,
        'zero_exp_avg_sq_tensors': 0,
    }

    for p in trainable:
        state = optimizer.state.get(p)
        if not state or not all(k in state for k in ('step', 'exp_avg', 'exp_avg_sq')):
            audit['missing_or_malformed_states'] += 1
            continue
        audit['states_checked'] += 1
        if _optimizer_step_as_int(state['step']) != EXPECTED_RESUME_UPDATE:
            audit['step_mismatches'] += 1
        if state['exp_avg'].shape != p.shape or state['exp_avg_sq'].shape != p.shape:
            audit['missing_or_malformed_states'] += 1
            continue
        if not bool((state['exp_avg'] != 0).any().item()):
            audit['zero_exp_avg_tensors'] += 1
        if not bool((state['exp_avg_sq'] != 0).any().item()):
            audit['zero_exp_avg_sq_tensors'] += 1

    failures = sum(audit[k] for k in (
        'missing_or_malformed_states', 'step_mismatches',
        'zero_exp_avg_tensors', 'zero_exp_avg_sq_tensors'
    ))
    assert audit['states_checked'] == len(trainable), 'D-096 BLOCKED: not every trainable parameter has AdamW state.'
    assert failures == 0, f'D-096 BLOCKED: AdamW continuity audit failed: {audit}'
    audit['passed'] = True
    return audit

# REQUIRED EXECUTION BEFORE TRAINING:
# optimizer_audit = audit_adamw_state(optimizer, resumed_model)


### 2.3 Epoch-4 shuffle continuity

The first extension epoch is epoch 4. Under the frozen epoch-indexed seed convention, its permutation must use **seed 46 (`42 + 4`)**. Record the first 10 example indices in `d096_resume_gate.json` before training so the data order can be independently reproduced.

In [ ]:
def epoch_permutation_indices(num_examples: int, epoch_number: int, base_seed: int = 42):
    import torch
    generator = torch.Generator()
    generator.manual_seed(base_seed + epoch_number)
    return torch.randperm(num_examples, generator=generator).tolist()

epoch4_indices = epoch_permutation_indices(CONTRACT.examples_per_epoch, epoch_number=4)
epoch4_first10 = epoch4_indices[:10]
assert EPOCH4_SHUFFLE_SEED == 46
assert len(epoch4_indices) == CONTRACT.examples_per_epoch
assert len(set(epoch4_indices)) == CONTRACT.examples_per_epoch
print('Epoch-4 shuffle seed:', EPOCH4_SHUFFLE_SEED)
print('First 10 example indices:', epoch4_first10)

### 2.4 Remaining hard-gate checks and machine-readable record

The executable gate must additionally verify checkpoint counters, Model C's 33,497,600-parameter config, GradScaler state on the canonical T4/FP16 path, persisted RNG state, tokenizer/corpus/manifest hashes, and the D-072 validation-stream identity. These observations, plus the validation and optimizer audits above, are written to `results/extended_training/model_c/d096_resume_gate.json`.

The training loop must begin with an assertion that this artifact's top-level `gate_passed` is `true`. A failed or incomplete gate is not a warning; it is a stop condition.

In [ ]:
# Gate assembly template — populate only from verified runtime observations.
D096_GATE_PATH = Path('results/extended_training/model_c/d096_resume_gate.json')

required_gate_fields = {
    'checkpoint_update': EXPECTED_RESUME_UPDATE,
    'checkpoint_completed_epoch': EXPECTED_RESUME_EPOCH,
    'model_parameters': CONTRACT.model_parameters,
    'tokenizer_sha256': EXPECTED_TOKENIZER_SHA256,
    'corpus_token_stream_sha256': EXPECTED_CORPUS_SHA256,
    'corpus_manifest_sha256': EXPECTED_MANIFEST_SHA256,
    'epoch4_shuffle_seed': EPOCH4_SHUFFLE_SEED,
    'epoch4_first10_indices': epoch4_first10,
    'validation_targets': CONTRACT.validation_targets,
}

# Later, after all runtime checks:
# gate_record = {**required_gate_fields,
#     'resume_validation': resume_validation_audit,
#     'optimizer_state': optimizer_audit,
#     'grad_scaler_state_verified': True,
#     'rng_state_verified': True,
#     'notebook05_artifacts_verified': True,
#     'gate_passed': True,
# }
# D096_GATE_PATH.parent.mkdir(parents=True, exist_ok=True)
# D096_GATE_PATH.write_text(json.dumps(gate_record, indent=2), encoding='utf-8')
# assert gate_record['gate_passed'] is True


## 3. Precommitted Post-Training Evaluation Policy

This policy is frozen **before extension training**:

1. Select the best 06A checkpoint using validation loss only.
2. Freeze that checkpoint.
3. Score it **exactly once** on the official test split using the same D-092 procedure. Record the result only as exploratory extended-Model-C evidence; never add it to the frozen Notebook 05 A/B/C test table and never use it for retuning.
4. Run the same D-091 validation-derived prompts with temperature 0.8, top-p 0.9, 96 new tokens, and identical per-prompt seeds.
5. Preserve the Notebook 05 qualitative finding that better perplexity did not guarantee more topically stable samples. Any improvement by extended C is evidence consistent with undertraining contributing to the prior drift, not proof of a capacity effect.

## Pause point

D-096 is now drafted as a fail-closed resume/provenance gate. **No extension training has started.**

Before this gate can actually pass, the exact Notebook 05 evaluation artifacts generated by the completed notebook must be committed to their canonical repository paths. They must not be reconstructed from rounded summaries.

Once those artifacts and the persistent Model C checkpoint are available in the execution environment, the next meaningful chunk is to **execute D-096 only**: load the checkpoint, reproduce validation loss 3.684501, audit AdamW moments/steps, verify scaler/RNG/hashes/counters, record the epoch-4 permutation prefix, write `d096_resume_gate.json`, and pause again before update 3,664.